In [1]:
from pathlib import Path
import ase.io
from tqdm import tqdm

IN_TRAIN = "train_pool.extxyz"
IN_VAL   = "val_ood.extxyz"
IN_TEST  = "test_ood.extxyz"

OUT_TRAIN_R00 = "train_pool_r00.extxyz"
OUT_VAL_R00   = "val_ood_r00.extxyz"
OUT_TEST_R00  = "test_ood_r00.extxyz"

for f in [OUT_TRAIN_R00, OUT_VAL_R00, OUT_TEST_R00]:
    Path(f).unlink(missing_ok=True)

def is_r00(atoms):
    # 先优先看 info["run"]
    run = atoms.info.get("run", None)
    if run is not None:
        s = str(run).strip().lower()
        if s in {"0", "00", "r00"}:
            return True

    # 再退回看 source_file
    src = str(atoms.info.get("source_file", "")).lower()
    return "__r00__" in src or "_r00_" in src or src.endswith("r00.traj") or src.endswith("r00.extxyz")

def filter_r00(infile, outfile, desc):
    kept = 0
    total = 0
    for atoms in tqdm(ase.io.iread(infile, index=":"), desc=desc):
        total += 1
        if is_r00(atoms):
            ase.io.write(outfile, atoms, append=True)
            kept += 1
    return total, kept

n_train_total, n_train_keep = filter_r00(IN_TRAIN, OUT_TRAIN_R00, "Filtering train r00")
n_val_total,   n_val_keep   = filter_r00(IN_VAL,   OUT_VAL_R00,   "Filtering val r00")
n_test_total,  n_test_keep  = filter_r00(IN_TEST,  OUT_TEST_R00,  "Filtering test r00")

print({
    "train": f"{n_train_keep}/{n_train_total}",
    "val":   f"{n_val_keep}/{n_val_total}",
    "test":  f"{n_test_keep}/{n_test_total}",
})

Filtering train r00: 120000it [24:07, 82.91it/s] 
Filtering val r00: 15000it [03:05, 80.69it/s] 
Filtering test r00: 15000it [03:10, 78.61it/s] 

{'train': '40000/120000', 'val': '5000/15000', 'test': '5000/15000'}
